### Character-level text generation with an LSTM

This section (3 of 4 in the course) was missing its notebook -- `shakespeare.txt`
(the complete works, ~186,000 lines) was sitting in `data/` unused. Here we build the
classic **character-level language model**: predict the next character given the
previous `seq_length` characters, one character at a time, trained on Shakespeare.

Unlike the word-level models in section 4 (`introtollm.ipynb`), a character-level model
has a tiny, fixed vocabulary (about 65 distinct characters instead of tens of
thousands of words), so it is small enough to train from scratch here rather than
needing a pretrained checkpoint -- a useful contrast with the "load a huge pretrained
model" pattern the rest of this course leans on.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

#### Loading and encoding the text

In [ ]:
with open("data/shakespeare.txt", "r") as f:
    text = f.read()

print(f"{len(text):,} characters total")
print(text[:300])

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"vocabulary size: {vocab_size} distinct characters")
print("".join(chars))

encoded = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)
print(f"\nencoded shape: {encoded.shape}")

#### Building training sequences

Same chunking idea as `3. RNNs/handling_sequences.ipynb` used for the electricity
data: slide a fixed-length window over the sequence, and the target for each window is
just the input shifted one character to the right.

In [ ]:
SEQ_LENGTH = 100

class CharDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_length]
        y = self.data[idx + 1: idx + self.seq_length + 1]
        return torch.tensor(x), torch.tensor(y)


# Use a slice of the full text for a fast CodeAlong run; swap in `encoded` for the
# full corpus once you have a GPU and time to let it train properly.
subset = encoded[:200_000]
dataset = CharDataset(subset, SEQ_LENGTH)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

x_sample, y_sample = dataset[0]
print("input :", "".join(idx_to_char[i.item()] for i in x_sample[:50]))
print("target:", "".join(idx_to_char[i.item()] for i in y_sample[:50]))

#### The model: embedding + LSTM + linear output

Same three-piece pattern as `handling_sequences.ipynb`'s `LSTMNet`, adapted for
characters: an embedding layer turns each character index into a dense vector, the
LSTM tracks context across the sequence, and a final linear layer produces a
probability distribution over the next character.

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=256, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden


model = CharLSTM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model has {n_params:,} trainable parameters")

#### Training loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()

N_EPOCHS = 3   # keep this small for a CodeAlong; the loss keeps falling well past this

model.train()
for epoch in range(N_EPOCHS):
    total_loss = 0.0
    for batch_idx, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits, _ = model(x)
        # logits: (batch, seq_len, vocab_size) -> flatten to (batch*seq_len, vocab_size)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 200 == 0:
            print(f"epoch {epoch+1}/{N_EPOCHS}  batch {batch_idx}/{len(loader)}  "
                  f"loss {loss.item():.4f}")

    print(f"epoch {epoch+1} average loss: {total_loss / len(loader):.4f}")

#### Generating text

Sample one character at a time from the model's predicted distribution, feed it back in
as the next input, and repeat. `temperature` controls how "confident" the sampling is:
low temperature sticks close to the model's top prediction (more repetitive, more
grammatical); high temperature samples more broadly (more novel, more likely to go off
the rails).

In [ ]:
def generate(model, start_text, length=300, temperature=0.8):
    model.eval()
    chars_idx = [char_to_idx[ch] for ch in start_text]
    input_seq = torch.tensor([chars_idx]).to(device)
    hidden = None
    generated = start_text

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(input_seq, hidden)
            last_logits = logits[0, -1] / temperature
            probs = torch.softmax(last_logits, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            generated += idx_to_char[next_idx]
            input_seq = torch.tensor([[next_idx]]).to(device)

    return generated

print(generate(model, start_text="ROMEO: ", length=300, temperature=0.8))

After only 3 epochs on a 200,000-character slice, do not expect coherent Shakespeare --
expect plausible-looking word lengths, some real short words ("the", "and", "to"), and
roughly correct punctuation and capitalisation patterns (character names in capitals
followed by a colon, line breaks). That surface-level structure emerging from nothing
but "predict the next character" is the whole point of the exercise: the model has
learned *something* about English orthography and Shakespearean formatting purely from
character sequences, with no notion of words built in.

#### Comparing temperatures

In [ ]:
for temp in (0.3, 0.8, 1.3):
    print(f"--- temperature = {temp} ---")
    print(generate(model, start_text="ROMEO: ", length=150, temperature=temp))
    print()

Low temperature tends to loop (the same safe, high-probability phrase repeated);
high temperature produces more variety at the cost of more nonsense words and broken
formatting -- the same exploration/exploitation trade-off you will see again in
reinforcement learning and in LLM sampling parameters (`top_p`, `top_k`) elsewhere in
this course.

#### What to try next

* Train on the full `encoded` array instead of the 200k-character subset, for far
  longer (tens of epochs), on a GPU.
* Replace the LSTM with a GRU (`nn.GRU`) and compare training speed and generated-text
  quality, following the RNN vs LSTM vs GRU comparison from
  `2.IntermediateDeepLearningWithPytorch/3. RNNs/handling_sequences.ipynb`.
* Seed generation with a real Shakespeare line instead of a bare character name, and see
  how much more coherent the continuation looks when the model gets more context to
  work from.